# D3 - US Accidents: Phân lớp + Gom cụm

## 1. Nguồn - giấy phép - quy mô

| Mục | Thông tin |
|---|---|
| Dataset | US Accidents |
| Link | https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents |
| Đơn vị công bố | Moosavi et al. (GeoAI), qua Kaggle |
| Giấy phép | CC BY-NC-SA 4.0 |
| Ngày tải | 2024 |
| Quy mô công bố | khoảng 3.0 triệu vụ tai nạn, 48 thuộc tính |
| Kỹ thuật yêu cầu | **Phân lớp + Gom cụm** |

**Tri thức lĩnh vực:** dữ liệu mô tả tai nạn giao thông tại Hoa Kỳ theo thời gian, vị trí, thời tiết và hạ tầng. `Severity` là mục tiêu rời rạc phù hợp cho phân lớp; các thuộc tính thời gian, tọa độ và điều kiện môi trường có thể dùng để tìm nhóm tai nạn tương đồng bằng gom cụm.

## 2. Từ điển dữ liệu

| Thuộc tính | Ý nghĩa | Kiểu | Thang đo |
|---|---|---|---|
| `ID` | Mã vụ tai nạn | string | định danh |
| `Severity` | Mức độ nghiêm trọng từ 1 đến 4 | int | thứ hạng |
| `Start_Time`, `End_Time` | Thời điểm bắt đầu/kết thúc | datetime | thời gian |
| `Start_Lat`, `Start_Lng` | Vĩ độ/kinh độ điểm bắt đầu | float | tỷ lệ |
| `Distance(mi)` | Chiều dài ảnh hưởng | float | tỷ lệ |
| `Temperature(F)`, `Visibility(mi)` | Điều kiện thời tiết | float | tỷ lệ |
| `Weather_Condition` | Mô tả thời tiết | category | danh nghĩa |
| `Sunrise_Sunset`, `Day_of_Week` | Bối cảnh ngày/đêm và thứ | category | danh nghĩa |
| `Amenity`, `Bump`, `Crossing`, `Junction` | Đặc điểm hạ tầng xung quanh | bool | nhị phân |

Ma trận kỹ thuật: **D3 = Phân lớp + Gom cụm**.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, silhouette_score
ROOT = Path(r'C:\data-mining-project\vs-code-interface-review\bai1-du-lieu-tien-xu-ly\D3')
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'outputs'
FIG = OUT / 'figures'
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(exist_ok=True)
SAMPLE_ROWS = 200000
files = list(RAW.rglob('*.csv'))
if not files:
    raise FileNotFoundError(f'Không tìm thấy file CSV trong {RAW}')
accident_path = max(files, key=lambda p: p.stat().st_size)
df = pd.read_csv(accident_path, nrows=SAMPLE_ROWS, low_memory=False)
print('File:', accident_path, '| kích thước:', df.shape)


## 3. Khám phá dữ liệu, giá trị thiếu và ngoại lệ

Báo cáo thiếu được lưu đầy đủ; các biến số được rà soát bằng quy tắc Z-score. Không xóa ngoại lệ địa lý một cách máy móc vì tai nạn nghiêm trọng và điều kiện thời tiết cực đoan là tín hiệu nghiệp vụ.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False).rename('missing_count').to_frame()
missing['missing_rate'] = missing['missing_count'] / len(df)
missing.to_csv(OUT / 'missing_report.csv', encoding='utf-8-sig')
numeric_all = df.select_dtypes(include='number').columns
z = ((df[numeric_all] - df[numeric_all].mean()) / df[numeric_all].std()).abs()
outliers = pd.DataFrame({'attribute': numeric_all, 'count_z_gt_3': (z > 3).sum().values})
outliers.to_csv(OUT / 'outlier_report.csv', index=False, encoding='utf-8-sig')
display(missing.head(20))


## 4. Thêm và biến đổi thuộc tính

Tách giờ, tháng, thứ và thời lượng ảnh hưởng từ thời gian; chuyển các cờ hạ tầng sang số. Các biến này vừa có ý nghĩa nghiệp vụ vừa phù hợp cho mô hình.

In [ ]:
for col in ['Start_Time', 'End_Time']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
df['start_hour'] = df['Start_Time'].dt.hour
df['start_month'] = df['Start_Time'].dt.month
df['start_weekday'] = df['Start_Time'].dt.dayofweek
df['duration_minutes'] = (df['End_Time'] - df['Start_Time']).dt.total_seconds() / 60
flag_cols = [c for c in ['Amenity', 'Bump', 'Crossing', 'Junction', 'Railway', 'Stop', 'Traffic_Signal'] if c in df]
for col in flag_cols:
    df[col] = df[col].fillna(False).astype(int)
features = ['start_hour', 'start_month', 'start_weekday', 'duration_minutes', 'Distance(mi)', 'Temperature(F)', 'Visibility(mi)'] + flag_cols
features = [c for c in features if c in df.columns]
df[features + ['Severity']].to_csv(PROCESSED / 'accidents_features.csv', index=False, encoding='utf-8-sig')


## 5. Phân lớp mức độ tai nạn

Dùng mẫu tối đa 300.000 dòng để notebook chạy ổn định trên máy cá nhân. Nhãn `Severity` được giữ nguyên bốn lớp; trọng số lớp giúp giảm thiên lệch do mất cân bằng.

In [ ]:
model_df = df.dropna(subset=['Severity']).copy()
if len(model_df) > 300000:
    model_df = model_df.sample(300000, random_state=42)
X = model_df[features]
y = model_df['Severity'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num = X.select_dtypes(include='number').columns.tolist()
prep = ColumnTransformer([('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num)], remainder='drop')
models = {'Logistic Regression': LogisticRegression(max_iter=300, class_weight='balanced'), 'Random Forest': RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1, class_weight='balanced_subsample')}
results = []
for name, estimator in models.items():
    pipe = Pipeline([('prep', prep), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({'Model': name, 'Accuracy': accuracy_score(y_test, pred), 'Precision': precision_score(y_test, pred, average='weighted', zero_division=0), 'Recall': recall_score(y_test, pred, average='weighted', zero_division=0), 'F1': f1_score(y_test, pred, average='weighted', zero_division=0)})
pd.DataFrame(results).to_csv(OUT / 'classification_results.csv', index=False)
display(pd.DataFrame(results))


## 6. Gom cụm không giám sát

Gom cụm các tai nạn theo thời điểm, địa lý, thời tiết và hạ tầng. Silhouette được dùng để so sánh số cụm.

In [ ]:
cluster_df = df[features].dropna().copy()
if len(cluster_df) > 100000:
    cluster_df = cluster_df.sample(100000, random_state=42)
Z = StandardScaler().fit_transform(cluster_df)
scores = []
for k in range(2, 8):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(Z)
    sample_idx = np.random.RandomState(42).choice(len(Z), size=min(5000, len(Z)), replace=False)
    scores.append({'k': k, 'silhouette': silhouette_score(Z[sample_idx], labels[sample_idx])})
scores = pd.DataFrame(scores)
scores.to_csv(OUT / 'clustering_scores.csv', index=False)
best_k = int(scores.loc[scores['silhouette'].idxmax(), 'k'])
cluster_df['cluster'] = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(Z)
cluster_df.to_csv(PROCESSED / 'accident_clusters.csv', index=False)
display(scores)
display(cluster_df.groupby('cluster')[features].mean().round(2))
